In [ ]:
# ================================================================
#           HIERARCHICAL CROP RECOMMENDATION (15 GROUPS)
# Season Prediction  →  Season-Specific 15-Group Crop Prediction
# Uses CatBoost + NDVI/EVI + Soil + Weather Features
# ================================================================

import numpy as np
import pandas as pd
import warnings, os
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# ----------------------- Install CatBoost -----------------------
try:
    from catboost import CatBoostClassifier
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier

SEED = 42
DATA_PATH = "./final_dataset_with_ndvi_15groups.csv"   # IMPORTANT
warnings.filterwarnings("ignore")

# --------------------------- Helpers ----------------------------
def topk_accuracy(y_true, proba, classes, k=3):
    idx = np.argsort(-proba, axis=1)[:, :k]
    topk = classes[idx]
    y = np.asarray(y_true)
    return np.mean([y[i] in topk[i] for i in range(len(y))])

def safe_split(X, y):
    try:
        return train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    except:
        return train_test_split(X, y, test_size=0.2, random_state=SEED)

def train_cb(X_tr, y_tr, X_val, y_val, cat_idx=[], verbose=200):
    model = CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.03,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        auto_class_weights="Balanced",
        random_seed=SEED,
        l2_leaf_reg=6,
        bootstrap_type="Bernoulli",
        subsample=0.7,
        grow_policy="Lossguide",
        verbose=verbose,
        early_stopping_rounds=100,
    )

    fit_args = {"eval_set": (X_val, y_val)}
    if len(cat_idx) > 0:
        fit_args["cat_features"] = cat_idx

    model.fit(X_tr, y_tr, use_best_model=True, **fit_args)
    return model

# ================================================================
#                     LOAD & PREPARE DATA
# ================================================================
df = pd.read_csv(DATA_PATH)

season_col = "Season"
crop_col = "Crop_Group_15"     # USE NEW 15-GROUP COLUMN

# Remove leakage columns
df = df.drop(columns=[c for c in ["Area", "Production"] if c in df.columns], errors="ignore")

valid_seasons = ["Kharif", "Rabi", "Zaid", "Annual"]
df = df[df[season_col].isin(valid_seasons)].copy()

# ================================================================
#                   SEASON PREDICTION MODEL
# ================================================================
season_cats = [c for c in ["state", "district"] if c in df.columns]
season_nums = df.select_dtypes(include=[np.number]).columns.tolist()

season_feats = [c for c in (season_cats + season_nums) if c not in [season_col, crop_col]]

X_season = df[season_feats]
y_season = df[season_col].astype(str)

X_se_tr, X_se_te, y_se_tr, y_se_te = safe_split(X_season, y_season)

cat_idx_season = [list(X_season.columns).index(c) for c in season_cats]
season_model = train_cb(X_se_tr, y_se_tr, X_se_te, y_se_te, cat_idx_season)

# ---------- Season Metrics ----------
season_pred = season_model.predict(X_se_te).ravel()
season_proba = season_model.predict_proba(X_se_te)
season_classes = np.array(season_model.classes_)

print("\n===== SEASON MODEL =====")
print("Accuracy :", round(accuracy_score(y_se_te, season_pred), 4))
print("Macro-F1 :", round(f1_score(y_se_te, season_pred, average="macro"), 4))
print("Top-3    :", round(topk_accuracy(y_se_te, season_proba, season_classes, 3), 4))

season_model.save_model("cb_season_predictor.cbm")

# ================================================================
#                 SEASON-SPECIFIC 15-GROUP CROP MODELS
# ================================================================
crop_models = {}
crop_class_map = {}

print("\n================================================")
print("           TRAINING SEASON-WISE CROP MODELS")
print("================================================")

for s in valid_seasons:
    df_s = df[df[season_col] == s]
    if df_s.empty:
        print(f"⚠ No rows for season: {s}")
        continue

    crop_cats = [c for c in ["state", "district"] if c in df_s.columns]
    crop_nums = df_s.select_dtypes(include=[np.number]).columns.tolist()

    crop_feats = [c for c in (crop_cats + crop_nums) if c != crop_col]

    Xc = df_s[crop_feats]
    yc = df_s[crop_col].astype(str)

    Xc_tr, Xc_te, yc_tr, yc_te = safe_split(Xc, yc)

    cat_idx = [list(Xc.columns).index(c) for c in crop_cats]
    model = train_cb(Xc_tr, yc_tr, Xc_te, yc_te, cat_idx)

    crop_models[s] = model
    crop_class_map[s] = np.array(model.classes_)

    # Metrics
    pred = model.predict(Xc_te).ravel()
    proba = model.predict_proba(Xc_te)

    print(f"\n===== {s.upper()} 15-GROUP CROP MODEL =====")
    print("Accuracy :", round(accuracy_score(yc_te, pred), 4))
    print("Macro-F1 :", round(f1_score(yc_te, pred, average='macro'), 4))
    print("Top-3    :", round(topk_accuracy(yc_te, proba, crop_class_map[s], 3), 4))

    model.save_model(f"cb_crop_{s.lower()}_15groups.cbm")

# ================================================================
#            END-TO-END (SEASON → CROP) EVALUATION
# ================================================================
print("\n================================================")
print("     END-TO-END (SEASON → 15-GROUP CROP) TESTING")
print("================================================")

df_test = df.loc[X_se_te.index]
true_crops = df_test[crop_col].astype(str).values
true_s = df_test[season_col].astype(str).values

top1 = top3 = 0
top1_o = top3_o = 0

for i, idx in enumerate(X_se_te.index):
    predicted_season = season_pred[i]

    row = df_test.loc[idx]

    # Predicted-season crop model
    if predicted_season in crop_models:
        feats = crop_models[predicted_season]

    df_s = df[df[season_col] == predicted_season]
    crop_cats = [c for c in ["state", "district"] if c in df_s.columns]
    crop_nums = df_s.select_dtypes(include=[np.number]).columns.tolist()
    feat_cols = [c for c in (crop_cats + crop_nums) if c != crop_col]

    row_input = pd.DataFrame([row[feat_cols].tolist()], columns=feat_cols)
    model_s = crop_models[predicted_season]
    classes = crop_class_map[predicted_season]

    proba = model_s.predict_proba(row_input)
    pred1 = classes[np.argmax(proba)]
    pred_top3 = classes[np.argsort(-proba)[0][:3]]

    if true_crops[i] == pred1:
        top1 += 1
    if true_crops[i] in pred_top3:
        top3 += 1

    # Oracle (true-season)
    true_season = true_s[i]
    if true_season in crop_models:
        df_s2 = df[df[season_col] == true_season]
        crop_cats2 = [c for c in ["state", "district"] if c in df_s2.columns]
        crop_nums2 = df_s2.select_dtypes(include=[np.number]).columns.tolist()
        feat_cols2 = [c for c in (crop_cats2 + crop_nums2) if c != crop_col]

        row2 = pd.DataFrame([row[feat_cols2].tolist()], columns=feat_cols2)

        model_t = crop_models[true_season]
        classes2 = crop_class_map[true_season]
        proba2 = model_t.predict_proba(row2)

        pred1_2 = classes2[np.argmax(proba2)]
        pred3_2 = classes2[np.argsort(-proba2)[0][:3]]

        if true_crops[i] == pred1_2:
            top1_o += 1
        if true_crops[i] in pred3_2:
            top3_o += 1

N = len(X_se_te)

print("\n===== END-TO-END RESULTS =====")
print(f"Top-1 Accuracy: {top1/N:.4f}")
print(f"Top-3 Accuracy: {top3/N:.4f}")

print("\n===== ORACLE (True Season) UPPER BOUND =====")
print(f"Top-1 Accuracy: {top1_o/N:.4f}")
print(f"Top-3 Accuracy: {top3_o/N:.4f}")

print("\nSaved all models successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.3 MB/s eta 0:00:00
0:	learn: 0.8875708	test: 0.8922086	best: 0.8922086 (0)	total: 329ms	remaining: 2m 44s
200:	learn: 0.9828171	test: 0.9809990	best: 0.9810763 (198)	total: 51.7s	remaining: 1m 16s
400:	learn: 0.9911530	test: 0.9891679	best: 0.9891679 (398)	total: 1m 34s	remaining: 23.4s
499:	learn: 0.9925035	test: 0.9900606	best: 0.9900606 (496)	total: 1m 57s	remaining: 0us

bestTest = 0.9900605616
bestIteration = 496

Shrink model to first 497 iterations.

===== SEASON MODEL =====
Accuracy : 0.9872
Macro-F1 : 0.9844
Top-3    : 1.0

           TRAINING SEASON-WISE CROP MODELS
0:	learn: 0.2671257	test: 0.2667006	best: 0.2667006 (0)	total: 422ms	remaining: 3m 30s
200:	learn: 0.3887565	test: 0.3803963	best: 0.3804819 (196)	total: 1m 26s	remaining: 2m 8s
400:	learn: 0.4120993	test: 0.3871168	best: 0.3876768 (346)	total: 2m 44s	remaining: 40.7s
499:	learn: 0.4316790	test: 0.3958279	best: 0.3967589 (487)	total: 3m 17s	remaining: 0us


In [ ]:
# ================================================================
#           VISUALIZATION CODE FOR RESULTS SECTION
# Generate publication-quality figures for research paper
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-paper')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

# ================================================================
#              FIGURE 1: SEASON PREDICTION CONFUSION MATRIX
# ================================================================

def plot_season_confusion_matrix(y_true, y_pred, save_path='figure1_season_confusion.png'):
    """
    Creates a 4x4 confusion matrix for season prediction

    Parameters:
    - y_true: True season labels
    - y_pred: Predicted season labels
    - save_path: Path to save the figure
    """
    # Define season order for better visualization
    seasons = ['Kharif', 'Rabi', 'Zaid', 'Annual']

    # Create confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=seasons)

    # Create figure
    fig, ax = plt.subplots(figsize=(8, 7))

    # Create display with percentages
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=seasons)
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=True)

    # Customize
    ax.set_title('Season Prediction Confusion Matrix', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Predicted Season', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Season', fontsize=12, fontweight='bold')

    # Add text annotations with percentages
    for i in range(len(seasons)):
        for j in range(len(seasons)):
            percentage = cm[i, j] / cm[i].sum() * 100
            text_color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j, i + 0.3, f'({percentage:.1f}%)',
                   ha='center', va='center', color=text_color, fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"✓ Saved: {save_path}")
    plt.close()

# ================================================================
#     FIGURE 2: THREE-PANEL CROP CONFUSION MATRICES
# ================================================================

def plot_crop_confusion_matrices(df, season_models, crop_col='Crop_Group_15',
                                 season_col='Season', save_path='figure2_crop_confusion.png'):
    """
    Creates side-by-side confusion matrices for Kharif, Rabi, and Annual crops

    Parameters:
    - df: Full dataset
    - season_models: Dictionary of trained season-specific models
    - crop_col: Name of crop group column
    - season_col: Name of season column
    - save_path: Path to save the figure
    """
    seasons = ['Kharif', 'Rabi', 'Annual']

    # Create figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    for idx, season in enumerate(seasons):
        ax = axes[idx]

        # Filter data for this season
        df_season = df[df[season_col] == season]

        if df_season.empty or season not in season_models:
            ax.text(0.5, 0.5, f'No data for {season}',
                   ha='center', va='center', fontsize=12)
            ax.set_title(f'{season} Season', fontsize=12, fontweight='bold')
            continue

        # Prepare features
        crop_cats = [c for c in ['state', 'district'] if c in df_season.columns]
        crop_nums = df_season.select_dtypes(include=[np.number]).columns.tolist()
        crop_feats = [c for c in (crop_cats + crop_nums) if c != crop_col]

        X = df_season[crop_feats]
        y_true = df_season[crop_col].astype(str)

        # Get predictions from saved model
        model = season_models[season]
        y_pred = model.predict(X).ravel()

        # Get unique labels for this season
        labels = sorted(df_season[crop_col].unique())

        # Create confusion matrix
        cm = confusion_matrix(y_true, y_pred, labels=labels)

        # Plot
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=90)

        # Customize
        ax.set_title(f'{season} Season\n(n={len(df_season):,})',
                    fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted Crop Group', fontsize=10)
        ax.set_ylabel('True Crop Group', fontsize=10)

        # Adjust tick labels
        ax.tick_params(axis='both', which='major', labelsize=7)

    plt.suptitle('Crop Group Confusion Matrices by Season',
                fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"✓ Saved: {save_path}")
    plt.close()

# ================================================================
#   FIGURE 3: PERFORMANCE COMPARISON BAR CHART
# ================================================================

def plot_performance_comparison(results_dict, save_path='figure3_performance_comparison.png'):
    """
    Creates grouped bar chart comparing Accuracy and F1-score across seasons

    Parameters:
    - results_dict: Dictionary with structure:
      {
        'Season': {'accuracy': 0.xx, 'f1': 0.xx, 'top3': 0.xx},
        'Kharif': {'accuracy': 0.xx, 'f1': 0.xx, 'top3': 0.xx},
        ...
      }
    - save_path: Path to save the figure
    """
    # Prepare data
    models = list(results_dict.keys())
    accuracy = [results_dict[m]['accuracy'] for m in models]
    f1_scores = [results_dict[m]['f1'] for m in models]
    top3_acc = [results_dict[m]['top3'] for m in models]

    # Set up bar positions
    x = np.arange(len(models))
    width = 0.25

    # Create figure
    fig, ax = plt.subplots(figsize=(12, 6))

    # Create bars
    bars1 = ax.bar(x - width, accuracy, width, label='Top-1 Accuracy',
                   color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x, f1_scores, width, label='Macro F1-Score',
                   color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=0.5)
    bars3 = ax.bar(x + width, top3_acc, width, label='Top-3 Accuracy',
                   color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=0.5)

    # Add value labels on bars
    def add_labels(bars):
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=8, fontweight='bold')

    add_labels(bars1)
    add_labels(bars2)
    add_labels(bars3)

    # Customize
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_xlabel('Model', fontsize=12, fontweight='bold')
    ax.set_title('Performance Comparison Across Season and Crop Models',
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10, fontweight='bold')
    ax.legend(loc='lower right', frameon=True, shadow=True)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"✓ Saved: {save_path}")
    plt.close()

# ================================================================
#              EXAMPLE USAGE WITH YOUR MODELS
# ================================================================

if __name__ == "__main__":
    # Load your data
    DATA_PATH = "/content/final_dataset_with_ndvi_15groups.csv"
    df = pd.read_csv(DATA_PATH)

    season_col = "Season"
    crop_col = "Crop_Group_15"

    # Filter valid seasons
    valid_seasons = ["Kharif", "Rabi", "Zaid", "Annual"]
    df = df[df[season_col].isin(valid_seasons)].copy()

    # ===================== FIGURE 1 =====================
    # Load season model and get predictions
    season_model = CatBoostClassifier()
    season_model.load_model("cb_season_predictor.cbm")

    # Prepare season features
    season_cats = [c for c in ["state", "district"] if c in df.columns]
    season_nums = df.select_dtypes(include=[np.number]).columns.tolist()
    season_feats = [c for c in (season_cats + season_nums) if c not in [season_col, crop_col]]

    X_season = df[season_feats]
    y_season_true = df[season_col].astype(str)

    # Get predictions
    y_season_pred = season_model.predict(X_season).ravel()

    # Plot Figure 1
    plot_season_confusion_matrix(y_season_true, y_season_pred)

    # ===================== FIGURE 2 =====================
    # Load crop models
    crop_models = {}
    for season in ['Kharif', 'Rabi', 'Annual']:
        try:
            model = CatBoostClassifier()
            model.load_model(f"cb_crop_{season.lower()}_15groups.cbm")
            crop_models[season] = model
        except:
            print(f"⚠ Could not load model for {season}")

    # Plot Figure 2
    plot_crop_confusion_matrices(df, crop_models)

    # ===================== FIGURE 3 =====================
    # Collect results (replace with your actual metrics)
    results = {
        'Season Prediction': {
            'accuracy': 0.9872,
            'f1': 0.9844,
            'top3': 1.0000
        },
        'Kharif Crops': {
            'accuracy': 0.3471,
            'f1': 0.3544,
            'top3': 0.7655
        },
        'Rabi Crops': {
            'accuracy': 0.5400,
            'f1': 0.5040,
            'top3': 0.9070
        },
        'Annual Crops': {
            'accuracy': 0.6352,
            'f1': 0.6677,
            'top3': 1.0000
        },
        'End-to-End': {
            'accuracy': 0.4581,
            'f1': 0.0,  # Not computed in original
            'top3': 0.8470
        }
    }

    # Plot Figure 3
    plot_performance_comparison(results)

    print("\n✅ All figures generated successfully!")
    print("📊 Figure 1: figure1_season_confusion.png")
    print("📊 Figure 2: figure2_crop_confusion.png")
    print("📊 Figure 3: figure3_performance_comparison.png")


✓ Saved: figure1_season_confusion.png
✓ Saved: figure2_crop_confusion.png
✓ Saved: figure3_performance_comparison.png

✅ All figures generated successfully!
📊 Figure 1: figure1_season_confusion.png
📊 Figure 2: figure2_crop_confusion.png
📊 Figure 3: figure3_performance_comparison.png
